In [ ]:
import pandas as pd
import numpy as np
import warnings
from arch import arch_model
from statsmodels.stats.diagnostic import het_arch

# Ignore convergence warnings for cleaner console output
warnings.filterwarnings('ignore')

def main():
    print("Loading data...")
    # Load the dataset
    df = pd.read_csv('../../master_dataset.csv', parse_dates=['date'], index_col='date')
    
    # Define sectors and combined price columns
    sectors = [
        'Banks', 'Capital_Goods', 'Consumer_Durables_and_Apparel', 
        'Consumer_Services', 'Diversified_Financials', 'Energy', 
        'Food,_Beverage_and_Tobacco', 'Materials', 'Real_Estate_Management_and_Development',
        'Retailing', 'Telecommunication_Services', 'Healthcare_Equipment_and_Services', 'Insurance'
    ]
    price_cols = ['ASPI'] + sectors
    
    print("Calculating log returns...")
    # Calculate log returns
    for col in price_cols:
        # We multiply by 100 to convert to percentage returns. 
        # This is critical for the ARCH model's maximum likelihood estimator to converge.
        df[f'{col}_Return'] = np.log(df[col] / df[col].shift(1)) * 100
        
    # Drop the first row since shifting creates NaNs
    df = df.dropna(subset=[f'{col}_Return' for col in price_cols])
    
    # Define the periods
    crisis_start, crisis_end = '2022-03-01', '2023-02-28'
    recovery_start, recovery_end = '2023-03-01', '2025-08-31'
    
    # Calculate pre-crisis end date (day before crisis starts)
    pre_crisis_end = (pd.to_datetime(crisis_start) - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
    
    # Slice the dataframe into the three distinct periods
    periods = {
        'Pre-Crisis': df.loc[:pre_crisis_end],
        'Crisis': df.loc[crisis_start:crisis_end],
        'Recovery': df.loc[recovery_start:recovery_end]
    }
    
    results = []

    print("Fitting EGARCH(1,1) models and running ARCH-LM tests...")
    # Loop through each period and each asset (ASPI + Sectors)
    for period_name, period_data in periods.items():
        for col in price_cols:
            # Extract the correct return column for the current asset
            y = period_data[f'{col}_Return'].dropna()
            
            # Skip if there is insufficient data for the period
            # if len(y) < 20:
            #     continue
                
            try:
                # Specify EGARCH(1,1)
                # In `arch`, p=1 (symmetric shock), o=1 (asymmetric shock/Gamma), q=1 (lagged volatility)
                am = arch_model(y, vol='EGARCH', p=1, o=1, q=1, dist='t')
                # res = am.fit(disp='off', options={'maxiter': 1000, 'ftol': 1e-4})
                res = am.fit(disp='off', options={'maxiter': 1000})              

                # Extract the Gamma parameter and its p-value
                gamma_param_name = [p for p in res.params.index if 'gamma' in p][0]
                gamma_val = res.params[gamma_param_name]
                gamma_pval = res.pvalues[gamma_param_name]

                # Extract the Alpha parameter
                alpha_param_name = [p for p in res.params.index if 'alpha' in p][0]
                alpha_val = res.params[alpha_param_name]

                # Extract the Beta parameter
                beta_param_name = [p for p in res.params.index if 'beta' in p][0]
                beta_val = res.params[beta_param_name]
                
                # Calculate standardized residuals to test for remaining ARCH effects
                std_resid = (res.resid / res.conditional_volatility).dropna()
                
                # Perform ARCH-LM test on standardized residuals (using standard 5 lags)
                arch_lm_test = het_arch(std_resid, nlags=5)
                arch_lm_pval = arch_lm_test[1] 
                
                # Store results
                results.append({
                    'Period': period_name,
                    'Asset': col,
                    'Alpha (Shock)': alpha_val,
                    'Beta (Persistence)': beta_val,
                    'Gamma (Leverage)': gamma_val,
                    'Gamma P-Value': gamma_pval,
                    'ARCH-LM P-Value': arch_lm_pval,
                    'Significant Leverage?': 'Yes' if (gamma_pval < 0.05 and gamma_val < 0) else 'No',
                    'Converged': res.convergence_flag == 0 
                })
                
            except Exception as e:
                print(f"Warning: Model failed for {col} in {period_name} - {e}")
                
    # Convert to DataFrame
    results_df = pd.DataFrame(results)
    
    # Sort for easier reading
    results_df['Period'] = pd.Categorical(
        results_df['Period'], 
        categories=['Pre-Crisis', 'Crisis', 'Recovery'], 
        ordered=True
    )
    results_df = results_df.sort_values(['Asset', 'Period']).reset_index(drop=True)
    
    print("\n--- Analysis Complete ---")
    print(results_df) 
    
    # Export results
    output_filename = 'egarch_1_1_results.csv'
    results_df.to_csv(output_filename, index=False)
    print(f"\nFull results successfully saved to {output_filename}")

if __name__ == "__main__":
    main()

Loading data...
Calculating log returns...
Fitting EGARCH(1,1) models and running ARCH-LM tests...

--- Analysis Complete ---
        Period                                   Asset  Alpha (Shock)  \
0   Pre-Crisis                                    ASPI       0.372607   
1       Crisis                                    ASPI       0.560170   
2     Recovery                                    ASPI       0.381424   
3   Pre-Crisis                                   Banks       0.419476   
4       Crisis                                   Banks       0.737145   
5     Recovery                                   Banks       0.264356   
6   Pre-Crisis                           Capital_Goods       0.315744   
7       Crisis                           Capital_Goods       0.601114   
8     Recovery                           Capital_Goods       0.324514   
9   Pre-Crisis           Consumer_Durables_and_Apparel       0.373829   
10      Crisis           Consumer_Durables_and_Apparel       0.351499  